In [ ]:
!pip install crewai

In [ ]:
!pip install crewai_tools

In [ ]:
!pip install textstat

In [ ]:
!pip install sentence_transformers

In [ ]:
!pip install qdrant-client

In [ ]:
!pip install PyPDF2

In [ ]:
!pip install gradio

In [ ]:
from crewai import Agent, Crew, Task, Process, LLM
from crewai_tools import FileWriterTool
from crewai.tools import tool

import requests
import csv
import os
import pandas as pd
import numpy as np
import re
import PyPDF2
import gradio as gr
import time

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from kaggle_secrets import UserSecretsClient
from IPython.display import Markdown, display
from bs4 import BeautifulSoup
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from typing import List

In [9]:
user_secrets = UserSecretsClient()

In [10]:
QDRANT_HOST = user_secrets.get_secret("QDRANT_HOST")
QDRANT_PORT = 6333
QDRANT_API_KEY = user_secrets.get_secret("QDRANT_API_KEY")

In [11]:
QDRANT_COLLECTION = "job_offers_collection"

In [12]:
apikey = user_secrets.get_secret("MISTRAL_API_KEY") 

In [13]:
api_hf = user_secrets.get_secret("HF_TOKEN")

In [ ]:
from crewai import BaseLLM
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

class CustomHuggingFaceLLM(BaseLLM):
    def __init__(self, model_name):
        super().__init__(model=model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)
        
    def call(self, messages, tools=None, callbacks=None, available_functions=None):
        # Convert messages to a single string
        if isinstance(messages, list):
            prompt = " ".join([m["content"] for m in messages])
        else:
            prompt = messages
            
        # Tokenize and generate
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        outputs = self.model.generate(
            **inputs,
            max_length=1000,
            pad_token_id=self.tokenizer.eos_token_id,
            do_sample=True,
            temperature = 0.1,
            
        )
        
        # Decode and return response
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response
        
    def supports_function_calling(self):
        return True
        
    def supports_stop_words(self):
        return False
        
    def get_context_window_size(self):
        return self.model.config.max_position_embeddings

# Usage example:
hf_llm = CustomHuggingFaceLLM("Qwen/Qwen2.5-1.5B-Instruct")

In [ ]:
sentence_transformer_model = SentenceTransformer('all-mpnet-base-v2')

In [16]:
custom_llm = LLM(
    model="mistral/mistral-large-2411",
    api_key=apikey,
    base_url="https://api.mistral.ai/v1",
    temperature=0.1
)

# Tools

In [17]:
@tool
def PDFReadTool(pdf_path: str) -> str:
    """This tool retrieve the content of a Pdf file"""
    # Apri il file PDF in modalità lettura binaria
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        
        # Leggi tutte le pagine
        testo = ''
        for pagina in reader.pages:
            testo += pagina.extract_text()
    
    return testo

In [18]:
@tool
def ScoreCVTool(skills: str) -> str:
    """This tool calculates a Cv score based on the candidate's number of skills"""
    # Divido la stringa sulle virgole
    list_skills = skills.split(',')
    n_skills = len(list_skills)
    score = str(2 ** n_skills)
    return {'Number of skills': n_skills, 'Score of Cv': score }

In [19]:
import textstat
import math

@tool
def TextReadabilityTool(cv_description: str) -> str:
    """This tool calculate the readability score of the Cv"""
    return {'readability_score': math.ceil(textstat.flesch_reading_ease(cv_description)) }

In [20]:
@tool
def QdrantSearchTool(cv_category: str, cv_text: str) -> str:
    "This tool is responsible for searching for job offers closest to the candidate's Cv text in the Qdrant vector database"
    
    # 1. Connessione al client
    client = QdrantClient(url=QDRANT_HOST, port=QDRANT_PORT, api_key = QDRANT_API_KEY)  
    
    # 2. Genero il vettore di query (cv_category + cv_description)
    query_text = f"{cv_category} {cv_text}"
    query_vector = sentence_transformer_model.encode([query_text], device='cuda')[0]
    
    # 3. Eseguo la ricerca nella collezione
    results = client.search(
        collection_name=QDRANT_COLLECTION,
        query_vector=query_vector,
        limit=5,
        with_payload=True  # per ottenere i metadati come title, description ecc.
    )
    
    # 4. Restituisco i risultati
    output = ""
    for hit in results:
        output+=f"\n**Title** : {hit.payload.get('title')}"
        output+=f"\n**Description** : {hit.payload.get('description')}"
        output+="\n########################################################################################################"
    
    return output

In [22]:
@tool
def CoherenceTool(text_1: str, text_2:str) -> str:
    "This tool is responsible for determining the lexical coherence between soft skills and hard skills in the resume"

    # Subroutine 1: Lexical Coherence 
    
    prompt1=(
    "Evaluate the lexical coherence only **between** the following two texts by providing a numerical (float) score between 0 and 1 with only two decimal places \n"
    "Text 1: \n"
    +text_1+"\n"
    "Text 2: \n"
    +text_2+"\n"

    )
    
    
    messages1 = [{"role": "user", "content": prompt1}]
    
    response = hf_llm.call(messages1)

    match = re.search(r"\b0\.\d{1,2}\b|\b1\.0{1,2}?\b", response)
    
    if match:
        lexical_coherence = float(match.group())
    else:
        return "Lexical coherence not find : " + response

    
    # Subroutine 2: Semantic Coherence 
    
    
    soft_skills = text_1.split(',')
    
    hard_skills = text_2.split(',')
    
    
    soft_embeddings = sentence_transformer_model.encode(soft_skills)
    hard_embeddings = sentence_transformer_model.encode(hard_skills)
    
    
    pairwise_sim = cosine_similarity(soft_embeddings, hard_embeddings)
    semantic_coherence = np.mean(pairwise_sim) # average_similarity
    

    # Combinazione lineare 

    a = 0.5
    
    total_coherence = a * lexical_coherence + (1-a) * semantic_coherence
    
    
    return {'Total Coherence': total_coherence, 'Lexical_coherence': lexical_coherence, 'Semantic coherence': semantic_coherence }


# Chatbot class

In [ ]:
import queue
import threading
import logging

class ChatBot:

    def my_step_callback(self, step):
        """Callback called after each agent step"""
        print(step)
        if "ToolResult" in str(step) and 'Tool Outputs' in self.check_buttons:
            self.step_messages.put("🤖 Tool Output: " + str(step.result))  # <- aggiunto alla coda
            self.step_messages.put("🤖 PLEASE WAIT ... ")
            self.step_messages.put("................................................................................................. ")
        #if "AgentAction" in str(step) and 'Reasoning Outputs' in self.check_buttons:
            #self.step_messages.put("🤖 Agent Reasoning: " + str(step.result))  # <- aggiunto alla coda
            #self.step_messages.put("🤖 PLEASE WAIT ... ")
            #self.step_messages.put("................................................................................................. ")

    def my_task_callback(self, task_output):
        if "Reasoning Plan:" in task_output.description and 'Reasoning Outputs' in self.check_buttons:
            start = task_output.description.rindex("Reasoning Plan:") # recupero l'indice della stringa "Reasoning Plan:" a partire da destra, perchè altrimenti catturerebbe uno dei "Reasoning Plan" all'interno del contesto
            self.step_messages.put("🤖 " + task_output.description[start:])  # <- aggiunto alla coda
            self.step_messages.put("🤖 PLEASE WAIT ... ")
            self.step_messages.put("................................................................................................. ")


    
    def __init__(self):
        self.step_messages = queue.Queue()  # <- coda per inviare step al generatore
        logging.basicConfig(level=logging.INFO)
        self.check_buttons=[]
            

        self.pdf_agent = Agent(
            role="Pdf Specialist",
            goal="Retrieve work experiences and skills of Cv's candidate from a path of pdf",
            backstory="""You are a human expert in the HR department of a forward-thinking recruitment agency.
        Your responsibility is to retrieve work experiences and skills in a file pdf of candidate's CV.""",
            allow_delegation=False,
            tools=[PDFReadTool],
            llm=custom_llm,
            
            
        )
        
        self.mathematical_stats_agent = Agent(
            role="Mathematical Specialist",
            goal="Analyze a CV and determine using custom tools the score of Cv and the readability score of CV",
            backstory="""You are a human expert in the HR department of a forward-thinking recruitment agency.
        Your responsibility is to evaluate mathematical statistics in a candidate's CV.""",
            allow_delegation=False,
            tools=[ScoreCVTool, TextReadabilityTool],
            llm=custom_llm,
            
            
        )
        self.job_offers_agent = Agent(
            role="Semantic Search Specialist",
            goal="Find job offers based on semantic search",
            backstory="You are an expert research assistant who can find job offers using semantic search in a Qdrant database.",
            tools=[QdrantSearchTool],
            llm=custom_llm,
            allow_delegation=False,
            
            
        )

        self.soft_skills_agent = Agent(
            role="Soft Skills Specialist",
            goal="Analyze a CV and determine how well its soft skills match job offer requirements",
            backstory="""You are a human expert in the HR department of a forward-thinking recruitment agency.
        Your responsibility is to analyze the soft skills listed in a candidate's CV and evaluate
        how well they match the soft skills required in job offers. You use a 1–10 scale to rate the
        compatibility between each CV and job offer, **strictly based on soft skills**.
        
        IMPORTANT:
        - Extract soft skills from the last Cv.
        - Then, read the last job offers (in the context).
        - You must clearly separate soft skills from hard skills in your reasoning.
        - You do not evaluate hard skills at all.""",
            allow_delegation=False,
            llm=custom_llm,
            
            
        )

        self.hard_skills_agent = Agent(
            role="Hard Skills Specialist",
            goal="Analyze a CV and determine how well its hard skills match job offer requirements",
            backstory="""You are a human expert in the HR department of a forward-thinking recruitment agency.
        Your responsibility is to analyze the hard skills listed in a candidate's CV and evaluate
        how well they match the hard skills required in job offers. You use a 1–10 scale to rate the
        compatibility between each CV and job offer, **strictly based on hard skills**.
        
        IMPORTANT:
        - Extract hard skills from the Cv.
        - Then, read the last job offers (in the context)
        - You must clearly separate hard skills from soft skills in your reasoning.
        - You do not evaluate soft skills at all.""",
            allow_delegation=False,
            llm=custom_llm,
            
            
        )

        self.skill_coherence_evaluator_agent = Agent(
            role="Skill Coherence Evaluator",
            goal="Determine the lexical and semantic coherence between soft skills e hard skills of Cv",
            backstory=(
                "In an era when the distinction between soft skills and hard skills is becoming increasingly blurred, the Agent leverages natural language processing to test whether a set of skills is consistent and applicable in real work scenarios."
                "Is able to measure the lexical and semantic distance between terms and assess whether a profile has a genuine balance between human and technical skills." 
                "It analyzes language patterns and realistic usage contexts."
            ),
            allow_delegation=False,
            tools=[CoherenceTool],
            llm=custom_llm,
            
            
        )

        self.salary_agent = Agent(
            role="Salary Estimation Specialist",
            goal="Determine the most appropriate salary based on compatibility scores and CV descriptions",
            backstory=(
                "You are an experienced HR and compensation specialist working in a global recruiting agency. "
                "You are responsible for evaluating candidates based on the compatibility scores given by other domain experts "
                "and assessing their seniority and background based on their CVs. "
                "You use a combination of data-driven logic and market standards to propose fair and competitive salaries."
                "So assign an appropriate salary in dollars ($) for each CV-job offer pairs"
            ),
            allow_delegation=False,
            llm=custom_llm,
            
            
        )
           
        
        self.chatbot_agent = Agent(
            role="Intelligent Assistant and Manager",
            goal="""Provide accurate and useful answers to users. 
                 Use available tools ONLY when strictly necessary 
                 to obtain updated or specific information you are not familiar with.""",
            backstory="""You are an experienced AI assistant who can answer a wide 
             range of questions. You have access to several tools, but you use them 
             sparingly--only when your knowledge is 
             insufficient or when the user requires very 
             specific or up-to-date information.""",
            max_iter=15, # per evitare loop
            allow_delegation = True,
            llm = custom_llm,
            
            
        )
    
        self.crew = Crew(
            agents=[self.pdf_agent, self.mathematical_stats_agent, self.job_offers_agent, self.soft_skills_agent, self.hard_skills_agent, self.skill_coherence_evaluator_agent, self.salary_agent],
            manager_agent=self.chatbot_agent,
            task_callback=self.my_task_callback,
            step_callback=self.my_step_callback,
        )
        
    
    def chat(self, context: str, user_input: str) -> str:
        """Handles a chat interaction"""
        
        task = Task(
            description=f"""
            the context is:
            "{context}"
            
            The user asked this question: "{user_input}"

            Remember: If the context is empty ([]) or the user's question requires knowing something that is not in the context, you need to make sure that you are not making things up, but answer that it is not possible to retrieve that particular information from the context.
            
            As chat manager:
            1. Analyze user input
            2. If you can answer directly with your knowledge and if you can answer, do it directly without making any delegation to another agent
            3. Delegate to the appropriate agents, **only** if you need:
            
               # To reading pdf files: delegation to "Pdf Specialist", and then formats the text
                
               # To evaluate the Cv score: 
               delegation to the "Mathematical Specialist"
                His inputs are:
                - The candidate's CV (last CV text in the context)
                - Extracted skills from the CV
                
                # To retrieve job openings: delegate to "Semantic Search Specialist", by passing as input the last cv category (between: Designer, Digital Media, Engineering, Finance, Healthcare, Information technology, Public relations, Sales") and the last **entire** cv text (not only the description or summary)
               
                # To evaluate how well the candidate’s soft skills match each job offer's soft skill requirements: delegation to the "Soft Skills Specialist"
                
                # To evaluate how well the candidate’s hard skills match each job offer's hard skill requirements: delegation to the "Hard Skills Specialist" 
                
                # To obtain the coherence or lexical coherence or semantic coherence between soft and hard skills of the Cv: 
                delegation to the Skill Coherence Evaluator to obtain the coherence (total coherence) the between the last soft and hard skills of the Cv (retrieved from the context) by passing in input the list of soft skills and the list of hard skills (Use comma as separator of soft/hard skills).

                # To assign an appropriate salary in dollars ($) for each CV-job offer pairs: delegation to the Salary Estimation Specialist

                Remember: delegate only those agents that meet the user's demand, do not delegate agents that do not do what is required in the user's demand
                 
            4. Provide a complete and useful response to the user
            
            Remember: your goal is to provide the best possible response to the user, 

            Important: At the **beginning** of the response sign with your role and the list of delegated agents, for example: 🤖 Intelligent Assistant and Manager \n Delegated agents: \n - 🤖 Agent1 \n - 🤖 Agent2 ... or 🤖❌ if you have not delegated anyone \n A separator \n Response:

            """,
            expected_output="A clear and helpful answer to the user's question",
            agent=self.chatbot_agent
        )

        
    
        self.crew.tasks = [task]
        result = self.crew.kickoff()
        return result.raw
    
    def start_conversation(self, user_input, history, file, check_buttons):
        
        messages = []
        self.check_buttons=check_buttons

        if 'Reasoning Outputs' in self.check_buttons:
            self.chatbot_agent.reasoning=True
            self.chatbot_agent.max_reasoning_attempts=3


        
        else:
            self.chatbot_agent.reasoning=False
            self.chatbot_agent.max_reasoning_attempts=None
   

        def run_crew():
            try:
                text = user_input + (f" {file}" if file else "")
                response = self.chat(history, text)
                self.step_messages.put(response)
            except Exception as e:
                self.step_messages.put(f"❌ Error: {e}")
    
        thread = threading.Thread(target=run_crew)
        thread.daemon=True
        thread.start()
        
    
        while thread.is_alive() or not self.step_messages.empty():
            try:
                step_msg = self.step_messages.get(timeout=0.5)
                messages.append(step_msg)
                for i in range(len(messages[-1])):
                    time.sleep(0.005)
                    full_output = "\n".join(messages[:-1]) + "\n" + messages[-1][: i + 1] #per lo scorrimento del testo
                    yield full_output 
                  
            
            except queue.Empty:
                continue


# Utilizzo
if __name__ == "__main__":
    bot = ChatBot()

    with gr.Blocks() as demo:
        chatbot = gr.Chatbot(type="messages", placeholder="<div style='text-align:center;font-size: 18px;'> Ask Me Anything about CV and Jobs <br> <br> You can upload the Pdf of a Cv by typing the path in chat or uploading it in the \"Attach PDF\" section")
        
        with gr.Row():
            
            with gr.Column(scale=0):
                chck_btn = gr.CheckboxGroup(["Tool Outputs", "Reasoning Outputs"], label="Enable intermediate outputs")
                
                gr.Markdown(
                    """
                    # List of Available Agents:
                    <p style="font-size: 16px;">
                    - <b>Intelligent Assistant and Manager</b><br>
                    </p>
                        <p style="font-size: 13px;">
                        Manager agent who is responsible for providing an appropriate response to the user's demand and if necessary makes delegation to other agents
                        </p>
                    <p style="font-size: 16px;">
                    - <b>Pdf Specialist:</b><br>
                    </p>
                        <p style="font-size: 13px;">
                        Agent that provides the contents of a pdf file by passing as input the file path within the chat or in the “Attach PDF” section
                        </p>
                    <p style="font-size: 16px;">
                    - <b>Mathematical Specialist</b><br>
                    </p>
                        <p style="font-size: 13px;">
                        To evaluate the Cv score and the text readability score
                        </p>
                    <p style="font-size: 16px;">
                    - <b>Semantic Search Specialist</b><br>
                    </p>
                        <p style="font-size: 13px;">
                        To retrieve job offers that are related to the Cv (semantically) using Vector Database 
                        </p>
                    <p style="font-size: 16px;">
                    - <b>Soft Skills Specialist</b><br>
                    </p>
                        <p style="font-size: 13px;">
                        To establish a compatibility score between each Cv-Job Offer pair based on soft skills
                        </p>
                    <p style="font-size: 16px;">
                    - <b>Hard Skills Specialist</b><br>
                    </p>
                        <p style="font-size: 13px;">
                        To establish a compatibility score between each Cv-Job Offer pair based on hard skills
                        </p>
                    <p style="font-size: 16px;">
                    - <b>Skill Coherence Evaluator</b><br>
                    </p>
                        <p style="font-size: 13px;">
                        To obtain the coherence or lexical coherence or semantic coherence between soft and hard skills of the Cv
                        </p>
                    <p style="font-size: 16px;">
                    - <b>Salary Estimation Specialist</b><br>
                    </p>
                        <p style="font-size: 13px;">
                        To assign an appropriate salary in dollars ($) for each CV-job offer pairs
                        </p>
                    """
                )
            with gr.Column():
                file = gr.File(label="Attach PDF", file_types=[".pdf"], visible=True)    
                gr.ChatInterface(
                    bot.start_conversation,
                    description="<p style='font-size: 20px;'>🤖 Chatbot: Searching, Understanding, and Reviewing Jobs and Curricula",
                    type="messages",
                    additional_inputs=[file, chck_btn],
                    chatbot=chatbot,
                    save_history=True,
            )
    
    demo.launch(share=True)